In [7]:
import json
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

DEFAULT_OUTPUT_DIR = BASE_DIR / "embedding" / "acc" / "logreg_topk_val"
EPS = 1e-12

ENTITY_CONFIGS = {
    "jobs": {
        "label": "jobs",
        "default_data_file": BASE_DIR / "data" / "acc" / "audit_tax_accounting_jobs.csv",
        "default_embeddings_file": BASE_DIR / "embedding" / "acc" / "acc_jobs_embeddings.jsonl",
        "data_id_column": "uuid",
        "title_column": "title",
        "skills_column": "extracted_skills",
        "embedding_id_key": "job_id",
    },
    "courses": {
        "label": "courses",
        "default_data_file": BASE_DIR / "data" / "2025-2026_module_clean_with_prereq_skillsfuture.csv",
        "default_embeddings_file": BASE_DIR / "embedding" / "acc" / "acc_courses_embeddings.jsonl",
        "data_id_column": "moduleCode",
        "title_column": "title",
        "skills_column": "extracted_skills",
        "embedding_id_key": "course_code",
    },
}


# Setting up main functions


In [8]:
def ensure_file_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return ""
    return text


def parse_skill_values(value: Any) -> list[str]:
    text = normalize_text(value)
    if not text:
        return []

    if text.startswith("["):
        try:
            raw = json.loads(text)
        except json.JSONDecodeError:
            raw = []
        if isinstance(raw, list):
            return [normalize_text(item) for item in raw if normalize_text(item)]

    if "|" in text:
        return [part for part in (normalize_text(item) for item in text.split("|")) if part]

    return [text]


def load_entity_rows(entity: str, data_file: str | Path | None = None) -> dict[str, dict[str, Any]]:
    config = ENTITY_CONFIGS[entity]
    data_path = Path(data_file) if data_file else config["default_data_file"]
    df = pd.read_csv(data_path)

    required_columns = [
        config["data_id_column"],
        config["title_column"],
        config["skills_column"],
    ]
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{entity} file is missing required column(s): {', '.join(missing_columns)}")

    rows = {}
    for row in df.itertuples(index=False):
        record_id = normalize_text(getattr(row, config["data_id_column"]))
        if not record_id:
            continue
        rows[record_id] = {
            "record_id": record_id,
            "title": normalize_text(getattr(row, config["title_column"])),
            "skills": parse_skill_values(getattr(row, config["skills_column"])),
        }
    return rows


def load_embedding_rows(entity: str, embeddings_file: str | Path | None = None) -> list[dict[str, Any]]:
    config = ENTITY_CONFIGS[entity]
    embeddings_path = Path(embeddings_file) if embeddings_file else config["default_embeddings_file"]

    rows = []
    with embeddings_path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            record = json.loads(stripped)
            record_id = normalize_text(record.get(config["embedding_id_key"]))
            split = normalize_text(record.get("split"))
            embedding = record.get("embedding")
            if not record_id or not split or not isinstance(embedding, list):
                raise ValueError(
                    f"{entity} embedding record on line {line_number} is missing a valid id, split, or embedding."
                )
            rows.append(
                {
                    "record_id": record_id,
                    "title": normalize_text(record.get("title")),
                    "split": split,
                    "embedding": np.asarray(embedding, dtype=np.float32),
                }
            )
    return rows


def combine_rows(
    entity: str,
    embedding_rows: list[dict[str, Any]],
    entity_rows: dict[str, dict[str, Any]],
) -> list[dict[str, Any]]:
    combined = []
    missing_ids = []

    for record in embedding_rows:
        entity_row = entity_rows.get(record["record_id"])
        if entity_row is None:
            missing_ids.append(record["record_id"])
            continue
        combined.append(
            {
                "record_id": record["record_id"],
                "title": entity_row["title"] or record["title"],
                "split": record["split"],
                "skills": entity_row["skills"],
                "embedding": record["embedding"],
            }
        )

    if missing_ids:
        preview = ", ".join(sorted(set(missing_ids))[:5])
        raise ValueError(
            f"Some {entity} embedding rows could not be matched to the source CSV. Examples: {preview}"
        )

    return combined


def select_split_rows(rows: list[dict[str, Any]], split_name: str) -> list[dict[str, Any]]:
    return [row for row in rows if row["split"] == split_name]


def fit_label_binarizer(
    train_rows: list[dict[str, Any]],
    *,
    min_positive_train: int = 2,
):
    raw_binarizer = MultiLabelBinarizer()
    y_train_raw = raw_binarizer.fit_transform([row["skills"] for row in train_rows])
    raw_classes = list(raw_binarizer.classes_)
    train_counts = Counter(
        {
            skill: int(count)
            for skill, count in zip(raw_classes, y_train_raw.sum(axis=0), strict=True)
        }
    )

    kept_classes = [
        skill
        for skill in raw_classes
        if train_counts[skill] >= min_positive_train and train_counts[skill] < len(train_rows)
    ]
    dropped_classes = [skill for skill in raw_classes if skill not in kept_classes]

    if not kept_classes:
        raise ValueError(
            "No trainable skills remain after filtering. Try reducing min_positive_train."
        )

    kept_class_set = set(kept_classes)
    filtered_train_labels = [
        [skill for skill in row["skills"] if skill in kept_class_set]
        for row in train_rows
    ]

    binarizer = MultiLabelBinarizer(classes=kept_classes)
    y_train = binarizer.fit_transform(filtered_train_labels)
    train_counts = Counter(
        {
            skill: int(count)
            for skill, count in zip(binarizer.classes_, y_train.sum(axis=0), strict=True)
        }
    )
    return binarizer, y_train, train_counts, dropped_classes


def transform_labels(rows: list[dict[str, Any]], binarizer: MultiLabelBinarizer):
    kept_class_set = set(binarizer.classes_)
    filtered_labels = [
        [skill for skill in row["skills"] if skill in kept_class_set]
        for row in rows
    ]
    ignored_skills = sorted(
        {
            skill
            for row in rows
            for skill in row["skills"]
            if skill not in kept_class_set
        }
    )
    y = binarizer.transform(filtered_labels)
    counts = Counter(
        {
            skill: int(count)
            for skill, count in zip(binarizer.classes_, y.sum(axis=0), strict=True)
        }
    )
    return y, counts, ignored_skills


def fit_models(
    x_train: np.ndarray,
    y_train: np.ndarray,
    class_names: list[str],
    *,
    max_iter: int = 1000,
    random_state: int = 42,
):
    models = []
    summaries = []

    for class_index, skill_name in enumerate(class_names):
        target = y_train[:, class_index]
        positive_count = int(target.sum())
        negative_count = int(len(target) - positive_count)
        if positive_count == 0 or negative_count == 0:
            raise ValueError(f"Skill '{skill_name}' is not trainable after filtering.")

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=max_iter,
            random_state=random_state,
            solver="liblinear",
        )
        model.fit(x_train, target)
        models.append(model)
        summaries.append(
            {
                "skill": skill_name,
                "train_positive_count": positive_count,
                "train_negative_count": negative_count,
                "coef_l2_norm": float(np.linalg.norm(model.coef_)),
                "intercept": float(model.intercept_[0]),
            }
        )

    return models, pd.DataFrame(summaries)


def predict_probabilities(models: list[LogisticRegression], x_data: np.ndarray) -> np.ndarray:
    probabilities = np.zeros((len(x_data), len(models)), dtype=np.float32)
    for class_index, model in enumerate(models):
        probabilities[:, class_index] = model.predict_proba(x_data)[:, 1]
    return probabilities


def _micro_metrics_from_counts(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 0.0 if precision + recall == 0.0 else (2 * precision * recall) / (precision + recall)
    return precision, recall, f1


def _is_better_top_k_candidate(candidate, incumbent) -> bool:
    c_f1, c_recall, c_macro_f1, c_k = candidate
    i_f1, i_recall, i_macro_f1, i_k = incumbent

    if c_f1 > i_f1 + EPS:
        return True
    if i_f1 > c_f1 + EPS:
        return False
    if c_recall > i_recall + EPS:
        return True
    if i_recall > c_recall + EPS:
        return False
    if c_macro_f1 > i_macro_f1 + EPS:
        return True
    if i_macro_f1 > c_macro_f1 + EPS:
        return False
    return c_k < i_k


def evaluate_with_top_k(
    rows: list[dict[str, Any]],
    class_names: list[str],
    y_true: np.ndarray,
    probabilities: np.ndarray,
    *,
    top_k: int,
):
    row_count = len(rows)
    num_classes = len(class_names)
    predictions = np.zeros((row_count, num_classes), dtype=bool)

    for row_index in range(row_count):
        if top_k <= 0 or num_classes == 0:
            continue
        limit = min(top_k, num_classes)
        top_indices = np.argsort(-probabilities[row_index])[:limit]
        predictions[row_index, top_indices] = True

    positives = y_true == 1
    tp_cols = np.logical_and(predictions, positives).sum(axis=0).astype(np.int64)
    fp_cols = np.logical_and(predictions, ~positives).sum(axis=0).astype(np.int64)
    fn_cols = np.logical_and(~predictions, positives).sum(axis=0).astype(np.int64)

    total_tp = int(tp_cols.sum())
    total_fp = int(fp_cols.sum())
    total_fn = int(fn_cols.sum())
    micro_precision, micro_recall, micro_f1 = _micro_metrics_from_counts(total_tp, total_fp, total_fn)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        predictions,
        average="macro",
        zero_division=0,
    )

    modeled_skill_set = set(class_names)
    per_row_metrics = []
    sample_precisions = []
    sample_recalls = []
    sample_f1_scores = []
    hit_flags = []
    exact_match_flags = []
    predicted_skill_counts = []
    true_skill_counts = []

    for row_index, row in enumerate(rows):
        true_skills = [skill for skill in row["skills"] if skill in modeled_skill_set]
        predicted_indices = np.flatnonzero(predictions[row_index])
        if predicted_indices.size:
            sort_order = np.argsort(-probabilities[row_index, predicted_indices])
            predicted_indices = predicted_indices[sort_order]

        predicted_skills = [class_names[index] for index in predicted_indices.tolist()]
        predicted_scores = [float(probabilities[row_index, index]) for index in predicted_indices]

        true_set = set(true_skills)
        predicted_set = set(predicted_skills)
        overlap = [skill for skill in predicted_skills if skill in true_set]

        precision = len(overlap) / len(predicted_skills) if predicted_skills else 0.0
        recall = len(overlap) / len(true_set) if true_set else 0.0
        f1_score = 0.0 if precision + recall == 0.0 else (2 * precision * recall) / (precision + recall)
        hit_flag = 1.0 if overlap else 0.0
        exact_match_flag = 1.0 if predicted_set == true_set else 0.0

        sample_precisions.append(precision)
        sample_recalls.append(recall)
        sample_f1_scores.append(f1_score)
        hit_flags.append(hit_flag)
        exact_match_flags.append(exact_match_flag)
        predicted_skill_counts.append(len(predicted_skills))
        true_skill_counts.append(len(true_skills))

        per_row_metrics.append(
            {
                "record_id": row["record_id"],
                "title": row["title"],
                "split": row["split"],
                "true_skills": " | ".join(true_skills),
                "predicted_top_k_skills": " | ".join(predicted_skills),
                "predicted_top_k_scores": json.dumps(
                    [
                        {"skill": skill, "probability": round(score, 6)}
                        for skill, score in zip(predicted_skills, predicted_scores, strict=True)
                    ]
                ),
                "correct_top_k_skills": " | ".join(overlap),
                "num_true_skills": len(true_skills),
                "num_predicted_skills": len(predicted_skills),
                "num_correct_top_k_skills": len(overlap),
                "precision_at_k": precision,
                "recall_at_k": recall,
                "f1_at_k": f1_score,
                "hit_at_k": hit_flag,
                "exact_match": exact_match_flag,
            }
        )

    metrics = {
        "top_k": int(top_k),
        "test_rows": len(rows),
        "candidate_skills": len(class_names),
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "sample_precision": float(np.mean(sample_precisions)) if sample_precisions else 0.0,
        "sample_recall": float(np.mean(sample_recalls)) if sample_recalls else 0.0,
        "sample_f1": float(np.mean(sample_f1_scores)) if sample_f1_scores else 0.0,
        "hit_rate": float(np.mean(hit_flags)) if hit_flags else 0.0,
        "exact_match_accuracy": float(np.mean(exact_match_flags)) if exact_match_flags else 0.0,
        "avg_true_skills_per_record": float(np.mean(true_skill_counts)) if true_skill_counts else 0.0,
        "avg_predicted_skills_per_record": (
            float(np.mean(predicted_skill_counts)) if predicted_skill_counts else 0.0
        ),
    }
    return metrics, pd.DataFrame(per_row_metrics)


def find_best_top_k(
    rows: list[dict[str, Any]],
    class_names: list[str],
    labels: np.ndarray,
    probabilities: np.ndarray,
    *,
    candidates: tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    best = None
    best_metrics = None

    for top_k in candidates:
        metrics, _ = evaluate_with_top_k(rows, class_names, labels, probabilities, top_k=top_k)
        candidate = (
            float(metrics["micro_f1"]),
            float(metrics["micro_recall"]),
            float(metrics["macro_f1"]),
            int(top_k),
        )
        if best is None or _is_better_top_k_candidate(candidate, best):
            best = candidate
            best_metrics = {
                "micro_precision": float(metrics["micro_precision"]),
                "micro_recall": float(metrics["micro_recall"]),
                "micro_f1": float(metrics["micro_f1"]),
                "macro_f1": float(metrics["macro_f1"]),
            }

    assert best is not None and best_metrics is not None
    return best[3], best_metrics


def build_top_k_df(
    class_names: list[str],
    *,
    selected_top_k: int,
    train_counts: Counter[str],
    val_counts: Counter[str],
    test_counts: Counter[str],
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "skill_name": class_names,
            "selected_top_k": [int(selected_top_k)] * len(class_names),
            "train_positive_support": [int(train_counts[skill]) for skill in class_names],
            "val_positive_support": [int(val_counts[skill]) for skill in class_names],
            "test_positive_support": [int(test_counts[skill]) for skill in class_names],
        }
    )


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")


def fit_acc_logreg_topk(
    entity: str = "jobs",
    data_file: str | Path | None = None,
    embeddings_file: str | Path | None = None,
    output_dir: str | Path | None = None,
    train_split: str = "train",
    val_split: str = "val",
    test_split: str = "test",
    top_k: int | None = None,
    top_k_candidates: tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
    min_positive_train: int = 2,
    max_iter: int = 1000,
    random_state: int = 42,
):
    config = ENTITY_CONFIGS[entity]
    data_path = Path(data_file) if data_file else config["default_data_file"]
    embeddings_path = Path(embeddings_file) if embeddings_file else config["default_embeddings_file"]
    output_root = Path(output_dir) if output_dir else DEFAULT_OUTPUT_DIR

    ensure_file_exists(data_path, f"{entity} file")
    ensure_file_exists(embeddings_path, f"{entity} embeddings file")

    entity_rows = load_entity_rows(entity, data_file=data_path)
    embedding_rows = load_embedding_rows(entity, embeddings_file=embeddings_path)
    combined_rows = combine_rows(entity, embedding_rows, entity_rows)

    split_counts = Counter(row["split"] for row in combined_rows)
    train_rows = select_split_rows(combined_rows, train_split)
    val_rows = select_split_rows(combined_rows, val_split)
    test_rows = select_split_rows(combined_rows, test_split)

    if not train_rows:
        raise ValueError(f"No rows found for train split '{train_split}'.")
    if not val_rows:
        raise ValueError(f"No rows found for val split '{val_split}'.")
    if not test_rows:
        raise ValueError(f"No rows found for test split '{test_split}'.")

    binarizer, y_train, train_counts, dropped_classes = fit_label_binarizer(
        train_rows,
        min_positive_train=min_positive_train,
    )
    y_val, val_counts, ignored_val_only_skills = transform_labels(val_rows, binarizer)
    y_test, test_counts, ignored_test_only_skills = transform_labels(test_rows, binarizer)

    class_names = list(binarizer.classes_)
    x_train = np.vstack([row["embedding"] for row in train_rows])
    x_val = np.vstack([row["embedding"] for row in val_rows])
    x_test = np.vstack([row["embedding"] for row in test_rows])

    models, trained_skills_df = fit_models(
        x_train,
        y_train,
        class_names,
        max_iter=max_iter,
        random_state=random_state,
    )

    val_probabilities = predict_probabilities(models, x_val)
    test_probabilities = predict_probabilities(models, x_test)

    if top_k is None:
        selected_top_k, top_k_selection_metrics = find_best_top_k(
            val_rows,
            class_names,
            y_val,
            val_probabilities,
            candidates=top_k_candidates,
        )
    else:
        selected_top_k = top_k
        top_k_selection_metrics, _ = evaluate_with_top_k(
            val_rows,
            class_names,
            y_val,
            val_probabilities,
            top_k=selected_top_k,
        )

    metrics, predictions_df = evaluate_with_top_k(
        test_rows,
        class_names,
        y_test,
        test_probabilities,
        top_k=selected_top_k,
    )
    top_k_df = build_top_k_df(
        class_names,
        selected_top_k=selected_top_k,
        train_counts=train_counts,
        val_counts=val_counts,
        test_counts=test_counts,
    )

    summary = {
        "entity": entity,
        "inputs": {
            "data_file": str(data_path),
            "embeddings_file": str(embeddings_path),
            "skills_column": config["skills_column"],
            "train_split": train_split,
            "val_split": val_split,
            "test_split": test_split,
        },
        "settings": {
            "prediction_mode": "top_k",
            "fixed_top_k": top_k,
            "top_k_candidates": list(top_k_candidates),
            "min_positive_train": min_positive_train,
            "max_iter": max_iter,
            "random_state": random_state,
        },
        "dataset": {
            "total_rows": len(combined_rows),
            "split_counts": dict(split_counts),
            "train_rows": len(train_rows),
            "val_rows": len(val_rows),
            "test_rows": len(test_rows),
            "trained_skills": len(class_names),
            "dropped_train_only_skills": dropped_classes,
            "ignored_val_only_skills": ignored_val_only_skills,
            "ignored_test_only_skills": ignored_test_only_skills,
        },
        "label_counts": {
            "train": dict(sorted(train_counts.items())),
            "val": dict(sorted(val_counts.items())),
            "test": dict(sorted(test_counts.items())),
        },
        "top_k_selection": {
            "split": val_split,
            "selected_top_k": int(selected_top_k),
            **top_k_selection_metrics,
        },
        "metrics": metrics,
    }

    metrics_df = pd.DataFrame(
        [
            {
                "entity": entity,
                "num_rows": len(combined_rows),
                "train_rows": len(train_rows),
                "val_rows": len(val_rows),
                "test_rows": len(test_rows),
                "num_skills": len(class_names),
                "train_split": train_split,
                "val_split": val_split,
                "test_split": test_split,
                "selected_top_k": selected_top_k,
                "val_selection_micro_precision": top_k_selection_metrics["micro_precision"],
                "val_selection_micro_recall": top_k_selection_metrics["micro_recall"],
                "val_selection_micro_f1": top_k_selection_metrics["micro_f1"],
                "val_selection_macro_f1": top_k_selection_metrics["macro_f1"],
                "micro_precision": metrics["micro_precision"],
                "micro_recall": metrics["micro_recall"],
                "micro_f1": metrics["micro_f1"],
                "macro_precision": metrics["macro_precision"],
                "macro_recall": metrics["macro_recall"],
                "macro_f1": metrics["macro_f1"],
                "sample_precision": metrics["sample_precision"],
                "sample_recall": metrics["sample_recall"],
                "sample_f1": metrics["sample_f1"],
                "hit_rate": metrics["hit_rate"],
                "exact_match_accuracy": metrics["exact_match_accuracy"],
                "avg_true_skills_per_record": metrics["avg_true_skills_per_record"],
                "avg_predicted_skills_per_record": metrics["avg_predicted_skills_per_record"],
                "num_dropped_train_only_skills": len(dropped_classes),
                "num_ignored_val_only_skills": len(ignored_val_only_skills),
                "num_ignored_test_only_skills": len(ignored_test_only_skills),
                "split_counts": json.dumps(dict(split_counts), sort_keys=True),
            }
        ]
    )

    output_root = Path(output_root)
    output_entity_dir = output_root / entity
    output_entity_dir.mkdir(parents=True, exist_ok=True)
    write_json(output_entity_dir / "metrics.json", summary)
    top_k_df.to_csv(output_entity_dir / "top_k_selection.csv", index=False)
    predictions_df.to_csv(output_entity_dir / "predictions.csv", index=False)
    trained_skills_df.to_csv(output_entity_dir / "trained_skills.csv", index=False)

    print(
        f"Evaluated {entity}: {len(train_rows)} train rows, {len(val_rows)} val rows, "
        f"{len(test_rows)} test rows, {len(class_names)} trained skills."
    )
    print(metrics_df.to_string(index=False, float_format=lambda value: f"{value:.4f}" if isinstance(value, float) else str(value)))

    return summary, metrics_df, top_k_df, predictions_df, trained_skills_df


# Final execution block


In [9]:
summary, metrics_df, top_k_df, predictions_df, trained_skills_df = fit_acc_logreg_topk(
    entity="jobs",
)

selection = summary["top_k_selection"]
metrics = summary["metrics"]
dataset = summary["dataset"]
inputs = summary["inputs"]

summary_md = f"""
## Validation results

- `ovr_top_k`
  - Validation split: `{inputs['val_split']}`
  - Selected top-k: `{selection['selected_top_k']}`
  - Micro Precision: `{selection['micro_precision']:.4f}`
  - Micro Recall: `{selection['micro_recall']:.4f}`
  - Micro F1: `{selection['micro_f1']:.4f}`
  - Macro F1: `{selection['macro_f1']:.4f}`

## Test results

- `ovr_top_k`
  - Test split: `{inputs['test_split']}`
  - Micro Precision: `{metrics['micro_precision']:.4f}`
  - Micro Recall: `{metrics['micro_recall']:.4f}`
  - Micro F1: `{metrics['micro_f1']:.4f}`
  - Macro Precision: `{metrics['macro_precision']:.4f}`
  - Macro Recall: `{metrics['macro_recall']:.4f}`
  - Macro F1: `{metrics['macro_f1']:.4f}`
  - Sample F1: `{metrics['sample_f1']:.4f}`
  - Hit rate: `{metrics['hit_rate']:.4f}`
  - Exact match accuracy: `{metrics['exact_match_accuracy']:.4f}`

## Dataset summary

- `Rows`
  - Train: `{dataset['train_rows']}`
  - Val: `{dataset['val_rows']}`
  - Test: `{dataset['test_rows']}`
  - Trained skills: `{dataset['trained_skills']}`
  - Dropped train-only skills: `{len(dataset['dropped_train_only_skills'])}`
  - Ignored val-only skills: `{len(dataset['ignored_val_only_skills'])}`
  - Ignored test-only skills: `{len(dataset['ignored_test_only_skills'])}`
"""

display(Markdown(summary_md))

display(Markdown("### Metrics table"))
display(metrics_df)

display(Markdown("### Top-k support preview"))
display(top_k_df.head(10))

display(Markdown("### Prediction preview"))
display(predictions_df.head(10))

display(Markdown("### Trained skill summary preview"))
display(trained_skills_df.head(10))

final_summary_md = f"""
### Summary

This notebook trains a one-vs-rest logistic regression model on the `{inputs['train_split']}` split using the precomputed embeddings and the `extracted_skills` labels. It then uses the `{inputs['val_split']}` split to choose a single global top-k value by comparing candidate k values and selecting the one with the best validation micro-F1, using recall, macro-F1, and smaller k as tie-breakers.

For this run, the model selected `top-k = {selection['selected_top_k']}` on validation. It was then evaluated once on the `{inputs['test_split']}` split, where it achieved micro-F1 `{metrics['micro_f1']:.4f}`, macro-F1 `{metrics['macro_f1']:.4f}`, and sample-F1 `{metrics['sample_f1']:.4f}`. The exact match accuracy was `{metrics['exact_match_accuracy']:.4f}`.

The dataset used `{dataset['train_rows']}` train rows, `{dataset['val_rows']}` validation rows, and `{dataset['test_rows']}` test rows, with `{dataset['trained_skills']}` trainable skills after filtering low-support labels. On average, the test rows had `{metrics['avg_true_skills_per_record']:.4f}` true skills and the model predicted `{metrics['avg_predicted_skills_per_record']:.4f}` skills per row.
"""

display(Markdown(final_summary_md))


Evaluated jobs: 59 train rows, 19 val rows, 20 test rows, 32 trained skills.
entity  num_rows  train_rows  val_rows  test_rows  num_skills train_split val_split test_split  selected_top_k  val_selection_micro_precision  val_selection_micro_recall  val_selection_micro_f1  val_selection_macro_f1  micro_precision  micro_recall  micro_f1  macro_precision  macro_recall  macro_f1  sample_precision  sample_recall  sample_f1  hit_rate  exact_match_accuracy  avg_true_skills_per_record  avg_predicted_skills_per_record  num_dropped_train_only_skills  num_ignored_val_only_skills  num_ignored_test_only_skills                         split_counts
  jobs        98          59        19         20          32       train       val       test               7                         0.6541                      0.7699                  0.7073                  0.4044           0.5714        0.7547    0.6504           0.2916        0.3742    0.3035            0.5714         0.7754     0.6327    1.0000      


## Validation results

- `ovr_top_k`
  - Validation split: `val`
  - Selected top-k: `7`
  - Micro Precision: `0.6541`
  - Micro Recall: `0.7699`
  - Micro F1: `0.7073`
  - Macro F1: `0.4044`

## Test results

- `ovr_top_k`
  - Test split: `test`
  - Micro Precision: `0.5714`
  - Micro Recall: `0.7547`
  - Micro F1: `0.6504`
  - Macro Precision: `0.2916`
  - Macro Recall: `0.3742`
  - Macro F1: `0.3035`
  - Sample F1: `0.6327`
  - Hit rate: `1.0000`
  - Exact match accuracy: `0.0000`

## Dataset summary

- `Rows`
  - Train: `59`
  - Val: `19`
  - Test: `20`
  - Trained skills: `32`
  - Dropped train-only skills: `20`
  - Ignored val-only skills: `2`
  - Ignored test-only skills: `6`


### Metrics table

,entity,num_rows,train_rows,val_rows,test_rows,num_skills,train_split,val_split,test_split,selected_top_k,...,sample_recall,sample_f1,hit_rate,exact_match_accuracy,avg_true_skills_per_record,avg_predicted_skills_per_record,num_dropped_train_only_skills,num_ignored_val_only_skills,num_ignored_test_only_skills,split_counts
0,jobs,98,59,19,20,32,train,val,test,7,...,0.775417,0.632695,1.0,0.0,5.3,7.0,20,2,6,"{""test"": 20, ""train"": 59, ""val"": 19}"


### Top-k support preview

...

### Prediction preview

,record_id,title,split,true_skills,predicted_top_k_skills,predicted_top_k_scores,correct_top_k_skills,num_true_skills,num_predicted_skills,num_correct_top_k_skills,precision_at_k,recall_at_k,f1_at_k,hit_at_k,exact_match
0,a227165cada83a2fb4f630cab69c76b4,Tax Assistant,test,Tax Computation | Tax Compliance,Tax Computation | Financial Reporting | Auditi...,"[{""skill"": ""Tax Computation"", ""probability"": 0...",Tax Computation | Tax Compliance,2,7,2,0.285714,1.000000,0.444444,1.0,0.0
1,4d15e5a23ca0edc50fbd3d41fc1356b7,ACCOUNTS ASSISTANT,test,Accounting Standards | Financial Transactions ...,Financial Transactions | Transactional Account...,"[{""skill"": ""Financial Transactions"", ""probabil...",Financial Transactions | Transactional Account...,5,7,5,0.714286,1.000000,0.833333,1.0,0.0
2,494eefdfc4c5b1f5ab083162041318c1,Account Assistant/Executive ( 1 Year Contract),test,Accounting Standards | Financial Reporting | F...,Transactional Accounting | Financial Transacti...,"[{""skill"": ""Transactional Accounting"", ""probab...",Transactional Accounting | Financial Transacti...,6,7,4,0.571429,0.666667,0.615385,1.0,0.0
3,e353ddde4dead8a0b51356c5117664c7,Audit Associate,test,Accounting Standards | Auditing and Assurance ...,Auditing and Assurance Standards | Engagement ...,"[{""skill"": ""Auditing and Assurance Standards"",...",Auditing and Assurance Standards | Engagement ...,4,7,2,0.285714,0.500000,0.363636,1.0,0.0
4,fcd54dc3bdff829d5c8a0181a3611f77,Accounts Executive - AP (MNC/ 5 Days / Up to $...,test,Transactional Accounting | Financial Transacti...,Financial Transactions | Transactional Account...,"[{""skill"": ""Financial Transactions"", ""probabil...",Financial Transactions | Transactional Account...,6,7,5,0.714286,0.833333,0.769231,1.0,0.0
5,0e5e6ee886bb2bdd400e4b2638b9b1fb,Accountant [Hougang | Up to $4300 | Full Set |...,test,Financial Reporting | Cash Flow Reporting | Ac...,Group Accounting and Consolidation | Cash Flow...,"[{""skill"": ""Group Accounting and Consolidation...",Group Accounting and Consolidation | Cash Flow...,10,7,7,1.000000,0.700000,0.823529,1.0,0.0
6,8f81bc1bbf31ba022153b1b8334387f8,Accounts & Admin Assistant #78279,test,Financial Transactions | Regulatory Compliance...,Financial Transactions | Financial Closing | T...,"[{""skill"": ""Financial Transactions"", ""probabil...",Financial Transactions | Financial Closing | T...,6,7,3,0.428571,0.500000,0.461538,1.0,0.0
7,bcacc19b763453f268b2ca87f9d14c27,Accounts Assistant,test,Accounting Standards | Transactional Accountin...,Financial Transactions | Transactional Account...,"[{""skill"": ""Financial Transactions"", ""probabil...",Financial Transactions | Transactional Account...,5,7,4,0.571429,0.800000,0.666667,1.0,0.0
8,1f458e526eec937e7a53ecfd35d9f8de,audit assistant,test,Accounting Standards | Internal Audit Engageme...,Financial Transactions | Financial Administrat...,"[{""skill"": ""Financial Transactions"", ""probabil...",Accounting Standards | Internal Controls,4,7,2,0.285714,0.500000,0.363636,1.0,0.0
9,b297e03d1cfdb79774b1312dec8a4fb9,Accounts Executive / Assistant (AP),test,Accounting Standards | Financial Transactions ...,Financial Transactions | Transactional Account...,"[{""skill"": ""Financial Transactions"", ""probabil...",Financial Transactions | Transactional Account...,4,7,4,0.571429,1.000000,0.727273,1.0,0.0


### Trained skill summary preview

,skill,train_positive_count,train_negative_count,coef_l2_norm,intercept
0,Accounting Standards,42,17,2.766932,-0.092968
1,Accounting and Tax Systems,8,51,3.396344,-0.410482
2,Audit Compliance,18,41,2.682897,-0.135924
3,Auditing and Assurance Standards,14,45,3.603771,-0.173820
4,Business Process Re-engineering,7,52,3.665629,-0.434496
5,Cash Flow Management,11,48,3.222576,-0.278604
6,Cash Flow Reporting,5,54,3.225343,-0.272848
7,Engagement Completion and Reporting,7,52,3.467145,-0.237421
8,Engagement Execution,14,45,3.635946,-0.193087
9,Engagement Planning,5,54,3.351938,-0.295351



### Summary

This notebook trains a one-vs-rest logistic regression model on the `train` split using the precomputed embeddings and the `extracted_skills` labels. It then uses the `val` split to choose a single global top-k value by comparing candidate k values and selecting the one with the best validation micro-F1, using recall, macro-F1, and smaller k as tie-breakers.

For this run, the model selected `top-k = 7` on validation. It was then evaluated once on the `test` split, where it achieved micro-F1 `0.6504`, macro-F1 `0.3035`, and sample-F1 `0.6327`. The exact match accuracy was `0.0000`.

The dataset used `59` train rows, `19` validation rows, and `20` test rows, with `32` trainable skills after filtering low-support labels. On average, the test rows had `5.3000` true skills and the model predicted `7.0000` skills per row.


To run the ACC course version instead, change the final cell to `entity="courses"`.
